In [1]:
import os 
import sys 

from pathlib import Path

sys.path.append(str(Path.cwd() / 'src'))

In [2]:
import options_last_price_api.main as api

In [3]:
import requests
import traceback
import json

# --- Configuration ---
# Update these values to match your server setup
API_BASE_URL = "http://dev.malctaylor15.com:9002"
API_KEY = "tk1"  # Replace with your actual SECRET_API_KEY

def test_health():
    """Checks if the API server is alive."""
    print("\n--- Testing Health Check ---")
    try:
        response = requests.get(f"{API_BASE_URL}/health")
        print(f"Status Code: {response.status_code}")
        print(f"Response: {response.json()}")
    except Exception as e:
        print(f"Error connecting to server: {e}")

def test_option_price(symbol):
    """Tests the /option-price endpoint with a Yahoo-style symbol."""
    print(f"\n--- Testing Option Price for: {symbol} ---")
    headers = {"X-API-Key": API_KEY}
    url = f"{API_BASE_URL}/option-price/{symbol}"
    
    try:
        response = requests.get(url, headers=headers)
        print(f"Status Code: {response.status_code}")
        print(json.dumps(response.json(), indent=2))
    except Exception as e:
        print(f"Request failed: {e}")

def test_earnings_date(identifier):
    """Tests the /earnings-date endpoint with a ticker or option symbol."""
    print(f"\n--- Testing Earnings Date for: {identifier} ---")
    headers = {"X-API-Key": API_KEY}
    url = f"{API_BASE_URL}/earnings-date/{identifier}"
    
    try:
        response = requests.get(url, headers=headers)
        print(f"Status Code: {response.status_code}")
        print(json.dumps(response.json(), indent=2))
    except Exception as e:
        print(f"Request failed: {e}")
        traceback.print_exc()

In [11]:

# 1. Check if server is up
test_health()

# 2. Test Option Price (Use standardized Yahoo format)
# Example: TSLA Dec 19 2025 $200 Call
test_option_price("MU260220C00055000")

# 3. Test Earnings Date (Works with Tickers or Option Symbols)
test_earnings_date("AAPL")            # Standard Ticker
test_earnings_date(".TSLA260206C455") # TOS Style Option
test_earnings_date("MU260220C00055000") # Yahoo Style Option


--- Testing Health Check ---
Status Code: 200
Response: {'status': 'ok', 'message': 'API is running.'}

--- Testing Option Price for: MU260220C00055000 ---
Status Code: 200
{
  "contract_symbol": "MU260220C00055000",
  "last_price": 141.48,
  "message": "Price fetched successfully."
}

--- Testing Earnings Date for: AAPL ---
Status Code: 200
{
  "ticker": "AAPL",
  "earnings_date": "2026-01-29",
  "days_to_earnings": 31,
  "message": "Successfully retrieved earnings date."
}

--- Testing Earnings Date for: .TSLA260206C455 ---
Status Code: 200
{
  "ticker": "TSLA",
  "earnings_date": "2026-01-28",
  "days_to_earnings": 30,
  "message": "Successfully retrieved earnings date."
}

--- Testing Earnings Date for: MU260220C00055000 ---
Status Code: 200
{
  "ticker": "MU",
  "earnings_date": "2025-12-17",
  "days_to_earnings": -12,
  "message": "Successfully retrieved earnings date."
}


## Check Earnings Date

In [12]:
ticker = "MU"

In [13]:
import re
import os
from datetime import datetime
from typing import Optional

from fastapi import FastAPI, Depends, HTTPException, status, Security
from fastapi.security import APIKeyHeader
from pydantic import BaseModel
import yfinance as yf
import pandas as pd

In [14]:
try:
    # Extract the base ticker in case an option symbol was passed
    base_ticker = api.parse_ticker_from_symbol(ticker)
    stock = yf.Ticker(base_ticker)

    # 1. Try stock.calendar
    calendar = stock.calendar
    if calendar is not None and calendar !={}:
        earnings_dates = calendar.get('Earnings Date')
        if earnings_dates is not None:
            next_date = earnings_dates[0]
            today = datetime.now().date()
            days_diff = (next_date - today).days

            out = {
                "ticker": base_ticker,
                "earnings_date": next_date.strftime('%Y-%m-%d'),
                "days_to_earnings": days_diff,
                "message": "Successfully retrieved earnings date." 
            }

    # 2. Fallback to .info metadata
    info_date = stock.info.get('earningsNext')
    if info_date:
        dt = datetime.fromtimestamp(info_date)
        today = datetime.now()
        days_diff = (dt.date() - today.date()).days
        out =  {
            "ticker": base_ticker,
            "earnings_date": dt.strftime('%Y-%m-%d'),
            "days_to_earnings": days_diff,
            "message": "Retrieved from info metadata."
        }

    out = {
        "ticker": base_ticker,
        "earnings_date": None,
        "days_to_earnings": None,
        "message": "No upcoming earnings date found."
    }
except Exception as e:
    print(e)
    traceback.print_exc()
    out =  {
        "ticker": ticker.upper(),
        "earnings_date": None,
        "days_to_earnings": None,
        "message": f"Error: {str(e)}"        
    }
    
print (out)

{'ticker': 'MU', 'earnings_date': None, 'days_to_earnings': None, 'message': 'No upcoming earnings date found.'}


In [15]:
earnings_dates

[datetime.date(2025, 12, 17)]

In [16]:
stock.calendar

{'Dividend Date': datetime.date(2026, 1, 13),
 'Ex-Dividend Date': datetime.date(2025, 12, 28),
 'Earnings Date': [datetime.date(2025, 12, 17)],
 'Earnings High': 9.07,
 'Earnings Low': 3.75,
 'Earnings Average': 8.05802,
 'Revenue High': 19340342000,
 'Revenue Low': 17375000000,
 'Revenue Average': 18722287790}